In [18]:
# -*- coding: utf-8 -*-
# 说明：端到端流程，包含模型预测、选股、回测与评估。
# 第 1 部分：基础依赖与通用工具
import os
import json
import math
import random
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from torch_geometric.data import Data
from torch_geometric.nn import RGCNConv
import torch.nn.functional as F


def set_seed(seed):
    """设置随机种子。"""
    torch.manual_seed(seed)
    random.seed(seed)
    np.random.seed(seed)


def get_device():
    """选择设备。"""
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("[状态] 设备 ->", device)
    return device

In [19]:
# 第 2 部分：图数据与特征处理
def load_returns_csv(path):
    """读取收益率矩阵。"""
    if not os.path.exists(path):
        raise FileNotFoundError('收益率文件不存在')
    df = pd.read_csv(path)
    date_col = 'trade_date' if 'trade_date' in df.columns else df.columns[0]
    dates = pd.to_datetime(df[date_col], errors='coerce')
    rets = df.drop(columns=[date_col], errors='ignore')
    rets.columns = [str(c) for c in rets.columns]
    rets = rets.apply(pd.to_numeric, errors='coerce')
    rets.index = dates
    return rets


def zscore_by_row(feat_df):
    """横截面标准化。"""
    mean = feat_df.mean(axis=0)
    std = feat_df.std(axis=0).replace(0, np.nan)
    return (feat_df - mean) / std


def build_features_from_window(window_df):
    """用历史窗口构造特征。"""
    mom20 = (1 + window_df.tail(20)).prod() - 1
    mean5 = window_df.tail(5).mean()
    mean10 = window_df.tail(10).mean()
    vol20 = window_df.tail(20).std(ddof=0)
    vol60 = window_df.tail(60).std(ddof=0)
    last_ret = window_df.tail(1).iloc[0]
    feat = pd.DataFrame({
        'mom20': mom20,
        'mean5': mean5,
        'mean10': mean10,
        'vol20': vol20,
        'vol60': vol60,
        'last_ret': last_ret
    })
    return zscore_by_row(feat)


def load_industry_mapping(path, stock_codes):
    """读取行业映射并对齐。"""
    if not os.path.exists(path):
        raise FileNotFoundError('行业映射文件不存在')
    df = pd.read_csv(path)
    df['ts_code'] = df['ts_code'].astype(str)
    df = df[df['ts_code'].isin(stock_codes)].copy()
    return df


def build_stock2idx(stock_codes):
    return {code: idx for idx, code in enumerate(stock_codes)}


def edge_index_from_pairs(edge_pairs):
    if not edge_pairs:
        return torch.empty((2, 0), dtype=torch.long)
    pair_tensor = torch.tensor(sorted(edge_pairs), dtype=torch.long)
    return pair_tensor.t().contiguous()


def build_industry_edges(industry_df, stock2idx, max_neighbors=20):
    edge_pairs = set()
    grouped = industry_df.groupby('industry')['ts_code'].apply(list).to_dict()
    for members in grouped.values():
        members = [m for m in members if m in stock2idx]
        for src in members:
            peers = [m for m in members if m != src]
            if len(peers) > max_neighbors:
                peers = peers[:max_neighbors]
            for dst in peers:
                edge_pairs.add((stock2idx[src], stock2idx[dst]))
    return edge_index_from_pairs(edge_pairs)


def build_corr_edges(window_df, top_k=20):
    data = window_df.to_numpy(dtype=float)
    if data.shape[0] < 5:
        return torch.empty((2, 0), dtype=torch.long)
    corr = np.corrcoef(data.T)
    edge_pairs = set()
    for i in range(corr.shape[0]):
        row = corr[i].copy()
        row[i] = -np.inf
        idx = np.argsort(row)[-top_k:]
        for j in idx:
            if np.isfinite(row[j]):
                edge_pairs.add((i, j))
    return edge_index_from_pairs(edge_pairs)

In [20]:
# 第 3 部分：模型与预测、选股
class StockRGCN_Regression(torch.nn.Module):
    """两层 RGCN 回归模型。"""
    def __init__(self, in_channels, hidden_channels, num_relations):
        super().__init__()
        self.conv1 = RGCNConv(in_channels, hidden_channels, num_relations)
        self.conv2 = RGCNConv(hidden_channels, hidden_channels, num_relations)
        self.regressor = torch.nn.Linear(hidden_channels, 1)

    def forward(self, x, edge_index, edge_type):
        x = self.conv1(x, edge_index, edge_type)
        x = F.relu(x)
        x = self.conv2(x, edge_index, edge_type)
        out = self.regressor(x).squeeze(-1)
        return out


def load_model(model_path, in_channels, hidden_channels, num_relations, device):
    """加载模型权重。"""
    if not os.path.exists(model_path):
        raise FileNotFoundError('模型文件不存在')
    print('[状态] 加载模型...')
    model = StockRGCN_Regression(in_channels, hidden_channels, num_relations).to(device)
    model.load_state_dict(torch.load(model_path, map_location=device))
    return model


def build_data_for_index(rets, industry_df, stock_codes, t_idx, lookback, top_k):
    """按日期索引构建图数据。"""
    window_df = rets.iloc[t_idx - lookback:t_idx]
    feat_df = build_features_from_window(window_df).reindex(stock_codes)
    x = torch.tensor(feat_df.to_numpy(dtype=float), dtype=torch.float)
    mask = ~torch.isnan(x).any(dim=1)
    stock2idx = build_stock2idx(stock_codes)
    ind_edge = build_industry_edges(industry_df, stock2idx, max_neighbors=20)
    corr_edge = build_corr_edges(window_df, top_k=top_k)
    edge_index = torch.cat([ind_edge, corr_edge], dim=1)
    edge_type = torch.cat([
        torch.zeros(ind_edge.size(1), dtype=torch.long),
        torch.ones(corr_edge.size(1), dtype=torch.long)
    ])
    data = Data(x=x, edge_index=edge_index, edge_type=edge_type)
    return data, mask


def predict_on_index(t_idx, model, rets, industry_df, stock_codes, lookback, top_k, device):
    """输出某日预测。"""
    data, mask = build_data_for_index(rets, industry_df, stock_codes, t_idx, lookback, top_k)
    model.eval()
    with torch.no_grad():
        out = model(data.x.to(device), data.edge_index.to(device), data.edge_type.to(device))
    pred = out.cpu().numpy()
    pred[~mask.cpu().numpy()] = np.nan
    return pd.DataFrame({'stock_code': stock_codes, 'pred_return': pred})


def select_stocks(pred_df, top_n=50):
    """按预测值选股。"""
    print('[状态] 执行选股...')
    pred_df = pred_df.dropna().sort_values('pred_return', ascending=False)
    selected = pred_df.head(top_n)['stock_code'].tolist()
    if len(selected) == 0:
        raise ValueError('选股结果为空')
    return selected

In [ ]:
# 第 4 部分：回测与评估
def compute_portfolio_return(returns_row, selected_codes, min_valid=10):
    """计算等权组合收益。"""
    valid = returns_row[selected_codes].dropna()
    if len(valid) < min_valid:
        return float('nan')
    return float(valid.mean())


def run_backtest(rets, industry_df, stock_codes, model, lookback, top_k, device, min_valid=10):
    """执行回测主循环。"""
    print('[状态] 开始回测...')
    daily_returns = []
    for t_idx in range(lookback, len(rets) - 1):
        pred_df = predict_on_index(t_idx, model, rets, industry_df, stock_codes, lookback, top_k, device)
        selected = select_stocks(pred_df)
        next_ret = compute_portfolio_return(rets.iloc[t_idx + 1], selected, min_valid=min_valid)
        if np.isnan(next_ret):
            print(f'[状态] {rets.index[t_idx+1].date()} 有效样本不足，跳过')
            continue
        daily_returns.append({'date': rets.index[t_idx + 1], 'portfolio_return': next_ret})
        print(f'[状态] {rets.index[t_idx+1].date()} 组合收益: {next_ret:.6f}')
    return pd.DataFrame(daily_returns)


def compute_drawdown(cum_returns):
    """计算最大回撤。"""
    peak = cum_returns.cummax()
    drawdown = (cum_returns - peak) / peak
    return float(drawdown.min())


def evaluate_performance(daily_ret_df):
    """输出核心指标。"""
    print('[状态] 评估绩效...')
    daily_returns = daily_ret_df['portfolio_return'].values
    if len(daily_returns) == 0:
        raise ValueError('收益序列为空')
    cum_returns = (1 + daily_returns).cumprod()
    annual_ret = cum_returns[-1] ** (252 / len(daily_returns)) - 1
    annual_vol = np.std(daily_returns) * math.sqrt(252)
    sharpe = annual_ret / annual_vol if annual_vol > 0 else 0.0
    max_drawdown = compute_drawdown(pd.Series(cum_returns))
    return {
        'annual_return': float(annual_ret),
        'annual_volatility': float(annual_vol),
        'sharpe_ratio': float(sharpe),
        'max_drawdown': float(max_drawdown)
    }

In [ ]:
# 第 5 部分：主函数入口
def run_pipeline():
    """执行完整流程并返回结果。"""
    returns_path = 'data/daily_returns.csv'  # 收益率路径
    industry_path = 'data/industry_mapping.csv'  # 行业映射路径
    model_path = 'best_model.pt'  # 模型路径
    hidden_channels = 64  # 隐层维度
    lookback = 60  # 回看窗口
    top_k = 20  # 相关性邻居数
    min_valid = 10  # 最少有效样本数
    seed = 42  # 随机种子

    set_seed(seed)
    device = get_device()
    rets = load_returns_csv(returns_path)
    stock_codes = list(rets.columns)
    industry_df = load_industry_mapping(industry_path, stock_codes)
    model = load_model(model_path, in_channels=6, hidden_channels=hidden_channels, num_relations=2, device=device)
    daily_ret_df = run_backtest(rets, industry_df, stock_codes, model, lookback, top_k, device, min_valid=min_valid)
    metrics = evaluate_performance(daily_ret_df)
    print('[状态] 回测完成')
    print(metrics)
    return daily_ret_df, metrics


if __name__ == '__main__':
    daily_ret_df, metrics = run_pipeline()

[状态] 设备 -> cuda
[状态] 加载模型...
[状态] 开始回测...


NameError: name 'device' is not defined

In [ ]:
# 第 6 部分：图表与反馈数据输出
def export_feedback(daily_ret_df, metrics, out_dir="outputs"):
    """保存反馈数据与图表。"""
    os.makedirs(out_dir, exist_ok=True)
    daily_ret_df = daily_ret_df.copy()
    daily_ret_df = daily_ret_df.dropna(subset=["portfolio_return"])
    daily_ret_df["cum_return"] = (1 + daily_ret_df["portfolio_return"]).cumprod()

    # 保存原始反馈数据
    daily_ret_path = os.path.join(out_dir, "daily_returns.csv")
    metrics_path = os.path.join(out_dir, "metrics.json")
    daily_ret_df.to_csv(daily_ret_path, index=False)
    with open(metrics_path, "w") as f:
        json.dump(metrics, f, ensure_ascii=False, indent=2)

    # 图 1：累计收益曲线
    plt.figure(figsize=(8, 4))
    plt.plot(daily_ret_df["date"], daily_ret_df["cum_return"], label="cum_return")
    plt.title("Cumulative Return")
    plt.xlabel("date")
    plt.ylabel("net value")
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, "cum_return.png"))

    # 图 2：日收益率曲线
    plt.figure(figsize=(8, 4))
    plt.plot(daily_ret_df["date"], daily_ret_df["portfolio_return"], label="daily_return")
    plt.title("Daily Return")
    plt.xlabel("date")
    plt.ylabel("return")
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, "daily_return.png"))

    # 图 3：回撤曲线
    cum_series = daily_ret_df["cum_return"]
    peak = cum_series.cummax()
    drawdown = (cum_series - peak) / peak
    plt.figure(figsize=(8, 4))
    plt.plot(daily_ret_df["date"], drawdown, label="drawdown")
    plt.title("Drawdown")
    plt.xlabel("date")
    plt.ylabel("drawdown")
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, "drawdown.png"))

    print("[状态] 已保存反馈数据与图表 ->", out_dir)
    return daily_ret_df


# 如果已运行主流程，可直接导出
try:
    export_feedback(daily_ret_df, metrics)
except Exception as e:
    print("[状态] 尚未生成回测结果，请先运行第 5 个单元", str(e))